# Multi-Modal Retrieval — CLIP → FAISS → BLIP rerank → **LoRA fine-tune**

Self-contained, interview-ready retrieval system. Upload this one notebook to Kaggle.

```
 query ─► CLIP bi-encoder ─► FAISS (Flat/IVF/HNSW) ─► top-K ─► BLIP cross-encoder rerank ─► top-K
                                                                         │
            Evaluation: R@K / MRR / mAP / nDCG  ± 95% bootstrap CI  +  paired significance
                                                                         │
            LoRA fine-tuning: parameter-efficient contrastive adaptation, tested before/after
```

**Runtime:** GPU **T4**, Internet **On**. **Dataset:** add `adityajn105/flickr8k`.
Configured for the **full dataset** (1,000-image test split). For a 2-minute sanity
pass instead, set `cfg.smoke_test=True` in the config cell.

## 0 · Dependencies (safeguards) + GPU check

In [ ]:
import os, warnings, logging, subprocess, sys, importlib
# --- silence noisy (harmless) warnings: HF hub token notice, transformers LOAD
#     REPORT / fast-processor notices, deprecation/future warnings ---
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("TRANSFORMERS_NO_ADVISORY_WARNINGS", "1")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
warnings.filterwarnings("ignore")
for _n in ["transformers", "huggingface_hub", "datasets", "faiss"]:
    logging.getLogger(_n).setLevel(logging.ERROR)

def _pip(*a): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a])

# FAISS isn't always preinstalled on Kaggle:
try: import faiss
except ImportError: _pip("faiss-cpu"); import faiss
# transformers floor with BLIP-ITM + LAION-CLIP support, then quiet its logger:
import transformers
from packaging.version import parse as _v
if _v(transformers.__version__) < _v("4.40"):
    _pip("transformers>=4.40")
    print("transformers upgraded — if imports error below, do Run → Restart & Run All")
transformers.logging.set_verbosity_error()
# PEFT (LoRA) and Gradio (demo):
for pkg in ["peft", "gradio"]:
    try: importlib.import_module(pkg)
    except ImportError: _pip(pkg)

import torch
print("torch", torch.__version__, "| CUDA",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
assert torch.cuda.is_available(), "Enable GPU: Settings → Accelerator → GPU T4 (needs phone-verified account)."


## 1 · The modular package
The next cells write the real `src/` modules to disk (single source of truth) — read
them as the system's source code. Each is also unit-tested with synthetic data.

In [ ]:
import os; os.makedirs("src", exist_ok=True)

In [ ]:
%%writefile src/utils.py
"""
utils.py
========
A context manager that silences BOTH Python-level (sys.stderr) and C-level
(file descriptor 2) writes. FAISS prints its clustering notices from C++ straight
to fd 2, which Python's `warnings`/`logging` cannot intercept — this can.
"""
from __future__ import annotations

import contextlib
import os


@contextlib.contextmanager
def suppress_stderr():
    """Redirect fd 2 and sys.stderr to /dev/null for the duration of the block.

    Python exceptions still propagate (their tracebacks print after the block
    exits and stderr is restored), so real errors are never hidden.
    """
    saved_fd = os.dup(2)
    devnull_fd = os.open(os.devnull, os.O_WRONLY)
    devnull_file = open(os.devnull, "w")
    try:
        os.dup2(devnull_fd, 2)
        with contextlib.redirect_stderr(devnull_file):
            yield
    finally:
        os.dup2(saved_fd, 2)
        os.close(saved_fd)
        os.close(devnull_fd)
        devnull_file.close()


In [ ]:
%%writefile src/interfaces.py
"""
interfaces.py
=============
The architectural backbone: small, explicit contracts that let every component
be swapped without touching the rest of the system. This is what turns a script
into a *system* -- you can change the encoder, the index type, or the reranker
independently, and the evaluation/serving code stays identical.

    Encoder      : raw images/text  -> L2-normalized embeddings
    VectorIndex  : embeddings       -> top-k nearest neighbours (exact OR approx)
    Reranker     : (query, candidates) -> reordered candidates (stage-2 scoring)

Everything downstream (pipeline, evaluation harness, API, demo) depends only on
these Protocols, never on a concrete CLIP/FAISS/BLIP class.
"""
from __future__ import annotations

from typing import List, Protocol, Sequence, Tuple, runtime_checkable

import numpy as np
from PIL import Image


@runtime_checkable
class Encoder(Protocol):
    """Maps a batch of images or texts into a shared, L2-normalized space."""
    name: str
    dim: int

    def encode_images(self, images: Sequence[Image.Image], batch_size: int = 64) -> np.ndarray: ...
    def encode_texts(self, texts: Sequence[str], batch_size: int = 256) -> np.ndarray: ...


@runtime_checkable
class VectorIndex(Protocol):
    """A searchable nearest-neighbour index over a fixed gallery."""
    name: str

    def build(self, embeddings: np.ndarray) -> "VectorIndex": ...
    def search(self, queries: np.ndarray, k: int) -> Tuple[np.ndarray, np.ndarray]:
        """Return (scores, indices), each shape (n_queries, k), best-first."""
        ...

    @property
    def ntotal(self) -> int: ...


@runtime_checkable
class Reranker(Protocol):
    """
    Stage-2 scorer. Given a query and the stage-1 candidate ids, return the
    candidates reordered best-first (optionally with new scores). A reranker may
    use a stronger, cross-modal model than the bi-encoder used for retrieval.
    """
    name: str

    def rerank_text_to_image(self, query_text: str,
                             candidate_ids: Sequence[int]) -> List[int]: ...
    def rerank_image_to_text(self, query_image: Image.Image,
                             candidate_ids: Sequence[int]) -> List[int]: ...


In [ ]:
%%writefile src/metrics.py
"""
metrics.py
==========
IR metrics computed as PER-QUERY arrays first, then aggregated with bootstrap
confidence intervals. Reporting a 95% CI (e.g. R@1 = 61.4 [59.8, 63.1]) instead
of a bare point estimate is the kind of rigor that stands out in an interview --
it says "I know my eval set is finite and my number has uncertainty."

Relevance is binary. Two query types are supported via `target_sets`:
  * single-target (text->image): each query has exactly one correct gallery item
  * multi-target (image->text):  each query has several correct gallery items

All functions take a FULL ranking (gallery indices, best-first) per query so that
MRR / mAP / nDCG see ranks beyond the recall cut-offs.
"""
from __future__ import annotations

from typing import Dict, List, Sequence

import numpy as np


# --------------------------------------------------------------------------- #
# Per-query primitives
# --------------------------------------------------------------------------- #
def first_hit_ranks(ranked_idx: np.ndarray, target_sets: Sequence[set]) -> np.ndarray:
    """1-indexed rank of the first relevant item per query (len+1 if none)."""
    ranks = np.full(len(ranked_idx), ranked_idx.shape[1] + 1, dtype=np.int64)
    for i, row in enumerate(ranked_idx):
        tg = target_sets[i]
        for r, j in enumerate(row, 1):
            if j in tg:
                ranks[i] = r
                break
    return ranks


def average_precision_row(row: np.ndarray, targets: set) -> float:
    if not targets:
        return 0.0
    hits, precs = 0, []
    for r, j in enumerate(row, 1):
        if j in targets:
            hits += 1
            precs.append(hits / r)
    return float(np.sum(precs) / len(targets)) if precs else 0.0


def ndcg_row(row: np.ndarray, targets: set, k: int) -> float:
    """Binary-relevance nDCG@k."""
    if not targets:
        return 0.0
    dcg = 0.0
    for pos, j in enumerate(row[:k], 1):
        if j in targets:
            dcg += 1.0 / np.log2(pos + 1)
    ideal = sum(1.0 / np.log2(i + 1) for i in range(1, min(len(targets), k) + 1))
    return float(dcg / ideal) if ideal > 0 else 0.0


# --------------------------------------------------------------------------- #
# Per-query metric bundle
# --------------------------------------------------------------------------- #
def per_query_metrics(ranked_idx: np.ndarray,
                      target_sets: Sequence[set],
                      ks: Sequence[int] = (1, 5, 10),
                      with_map: bool = True,
                      ndcg_k: int = 10) -> Dict[str, np.ndarray]:
    """Return a dict of per-query arrays (length = n_queries)."""
    ranks = first_hit_ranks(ranked_idx, target_sets)
    out: Dict[str, np.ndarray] = {f"R@{k}": (ranks <= k).astype(np.float64) for k in ks}
    out["MRR"] = 1.0 / ranks
    out[f"nDCG@{ndcg_k}"] = np.array(
        [ndcg_row(ranked_idx[i], target_sets[i], ndcg_k) for i in range(len(ranked_idx))])
    if with_map:
        out["mAP"] = np.array(
            [average_precision_row(ranked_idx[i], target_sets[i]) for i in range(len(ranked_idx))])
    return out


# --------------------------------------------------------------------------- #
# Aggregation with bootstrap confidence intervals
# --------------------------------------------------------------------------- #
def bootstrap_ci(values: np.ndarray, n_boot: int = 1000, alpha: float = 0.05,
                 seed: int = 0) -> tuple:
    """Percentile bootstrap CI for the mean of a per-query metric array."""
    values = np.asarray(values, dtype=np.float64)
    n = len(values)
    if n == 0:
        return (0.0, 0.0, 0.0)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, n, size=(n_boot, n))
    boot_means = values[idx].mean(axis=1)
    lo, hi = np.percentile(boot_means, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return (float(values.mean()), float(lo), float(hi))


def aggregate(per_query: Dict[str, np.ndarray], as_percent: bool = True,
              n_boot: int = 1000, seed: int = 0) -> Dict[str, dict]:
    """
    Collapse per-query arrays into {metric: {mean, lo, hi}} with bootstrap CIs.
    Recall/mAP/nDCG are scaled to percent; MRR stays in [0, 1].
    """
    result = {}
    for name, arr in per_query.items():
        mean, lo, hi = bootstrap_ci(arr, n_boot=n_boot, seed=seed)
        scale = 100.0 if (as_percent and name != "MRR") else 1.0
        result[name] = {"mean": round(mean * scale, 2),
                        "lo": round(lo * scale, 2),
                        "hi": round(hi * scale, 2)}
    return result


def format_row(direction: str, agg: Dict[str, dict]) -> str:
    parts = [f"{direction:<14}"]
    for name, v in agg.items():
        parts.append(f"{name}={v['mean']:.2f} [{v['lo']:.2f},{v['hi']:.2f}]")
    return "  ".join(parts)


In [ ]:
%%writefile src/indexes.py
"""
indexes.py
==========
Concrete VectorIndex implementations behind a single interface, so the pipeline
and evaluation code never care whether search is exact or approximate.

  FlatIPIndex : exact inner-product (cosine on normalized vectors). Optimal recall,
                the right choice at this scale (a few thousand vectors).
  IVFFlatIndex: inverted-file ANN. Trades a little recall for speed at large scale.
  HNSWIndex   : graph-based ANN. Strong speed/recall trade-off, no training.

The point of shipping all three is to *demonstrate* the exact-vs-approximate
trade-off (see results/ann_tradeoff.png from the notebook), then justify using
Flat for the actual benchmark because the gallery is small.
"""
from __future__ import annotations

from typing import Tuple

import numpy as np
import faiss


class _Base:
    name = "base"

    def __init__(self):
        self.index = None

    def _prep(self, x: np.ndarray) -> np.ndarray:
        return np.ascontiguousarray(x, dtype="float32")

    def search(self, queries: np.ndarray, k: int) -> Tuple[np.ndarray, np.ndarray]:
        k = min(k, self.ntotal)
        return self.index.search(self._prep(queries), k)

    @property
    def ntotal(self) -> int:
        return 0 if self.index is None else self.index.ntotal


class FlatIPIndex(_Base):
    name = "flat_ip"

    def build(self, embeddings: np.ndarray) -> "FlatIPIndex":
        emb = self._prep(embeddings)
        self.index = faiss.IndexFlatIP(emb.shape[1])
        self.index.add(emb)
        return self


class IVFFlatIndex(_Base):
    name = "ivf_flat"

    def __init__(self, nlist: int = 100, nprobe: int = 16):
        super().__init__()
        self.nlist, self.nprobe = nlist, nprobe

    def build(self, embeddings: np.ndarray) -> "IVFFlatIndex":
        from utils import suppress_stderr
        emb = self._prep(embeddings)
        d = emb.shape[1]
        # FAISS wants >= ~39*nlist training points; clamp nlist so it never
        # under-trains (which would emit a C-level clustering warning).
        nlist = max(1, min(self.nlist, emb.shape[0] // 39, emb.shape[0]))
        quantizer = faiss.IndexFlatIP(d)
        self.index = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_INNER_PRODUCT)
        with suppress_stderr():
            self.index.train(emb)
            self.index.add(emb)
        self.index.nprobe = min(self.nprobe, nlist)
        return self


class HNSWIndex(_Base):
    name = "hnsw"

    def __init__(self, M: int = 32, ef_search: int = 64):
        super().__init__()
        self.M, self.ef_search = M, ef_search

    def build(self, embeddings: np.ndarray) -> "HNSWIndex":
        emb = self._prep(embeddings)
        self.index = faiss.IndexHNSWFlat(emb.shape[1], self.M, faiss.METRIC_INNER_PRODUCT)
        self.index.hnsw.efSearch = self.ef_search
        self.index.add(emb)
        return self


def make_index(kind: str = "flat_ip", **kwargs):
    return {"flat_ip": FlatIPIndex,
            "ivf_flat": IVFFlatIndex,
            "hnsw": HNSWIndex}[kind](**kwargs)


In [ ]:
%%writefile src/encoders.py
"""
encoders.py
===========
Concrete Encoder built on a Hugging Face CLIP model. Implements the Encoder
interface so the rest of the system never references CLIP directly.

Model menu (only the string changes -- all load via transformers.CLIPModel):
    fast : openai/clip-vit-base-patch32        (baseline)
    b16  : openai/clip-vit-base-patch16
    best : laion/CLIP-ViT-L-14-laion2B-s32B-b82K   (recommended)
    max  : laion/CLIP-ViT-H-14-laion2B-s32B-b79K
"""
from __future__ import annotations

import time
from typing import Sequence

import numpy as np
import torch
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

from utils import suppress_stderr

MODELS = {
    "fast": "openai/clip-vit-base-patch32",
    "b16":  "openai/clip-vit-base-patch16",
    "best": "laion/CLIP-ViT-L-14-laion2B-s32B-b82K",
    "max":  "laion/CLIP-ViT-H-14-laion2B-s32B-b79K",
}


def _l2(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype="float32")
    n = np.linalg.norm(x, axis=1, keepdims=True)
    n[n == 0] = 1.0
    return x / n


class CLIPEncoder:
    """Encoder Protocol implementation."""

    def __init__(self, model_key: str = "best", device: str | None = None):
        self.name = MODELS.get(model_key, model_key)
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        t0 = time.time()
        with suppress_stderr():
            self.model = CLIPModel.from_pretrained(self.name).to(self.device).eval()
            self.processor = CLIPProcessor.from_pretrained(self.name)
        self.dim = self.model.config.projection_dim
        print(f"[CLIPEncoder] {self.name} on {self.device} | dim={self.dim} "
              f"| load {time.time()-t0:.1f}s")

    @torch.no_grad()
    def encode_images(self, images: Sequence[Image.Image], batch_size: int = 64) -> np.ndarray:
        out = []
        for i in range(0, len(images), batch_size):
            batch = [im.convert("RGB") for im in images[i:i + batch_size]]
            inp = self.processor(images=batch, return_tensors="pt").to(self.device)
            with torch.autocast(self.device):
                # Explicit projection path — stable across transformers versions
                # (get_image_features' return type changed in newer releases).
                vout = self.model.vision_model(pixel_values=inp["pixel_values"])
                feat = self.model.visual_projection(vout.pooler_output)
            out.append(feat.float().cpu().numpy())
        return _l2(np.concatenate(out, 0))

    @torch.no_grad()
    def encode_texts(self, texts: Sequence[str], batch_size: int = 256) -> np.ndarray:
        out = []
        for i in range(0, len(texts), batch_size):
            inp = self.processor(text=list(texts[i:i + batch_size]), return_tensors="pt",
                                 padding=True, truncation=True).to(self.device)
            with torch.autocast(self.device):
                tout = self.model.text_model(input_ids=inp["input_ids"],
                                             attention_mask=inp["attention_mask"])
                feat = self.model.text_projection(tout.pooler_output)
            out.append(feat.float().cpu().numpy())
        return _l2(np.concatenate(out, 0))


In [ ]:
%%writefile src/rerankers.py
"""
rerankers.py
============
Stage-2 rerankers implementing the Reranker interface. Given a query and the
stage-1 candidate ids, return them reordered best-first.

  IdentityReranker      : no-op. The honest ablation baseline ("rerank off").
  CLIPRescoreReranker   : rescore the head with a (typically larger) CLIP. Cheap,
                          no extra model family. "retrieve small, rerank large".
  BLIPReranker          : true cross-encoder. A BLIP image-text-matching (ITM)
                          head attends over image AND text jointly, which a
                          bi-encoder cannot -- this is the strongest reranker and
                          the architectural centerpiece.

Rerankers hold references to the gallery payloads (image paths aligned to the
image gallery, caption texts aligned to the text gallery) so they can fetch the
actual content for a candidate id.
"""
from __future__ import annotations

from typing import List, Sequence

import numpy as np
from PIL import Image


# --------------------------------------------------------------------------- #
class IdentityReranker:
    name = "identity"

    def rerank_text_to_image(self, query_text, candidate_ids): return list(candidate_ids)
    def rerank_image_to_text(self, query_image, candidate_ids): return list(candidate_ids)


# --------------------------------------------------------------------------- #
class CLIPRescoreReranker:
    """
    Recompute cosine similarity for the head using a (possibly different/larger)
    CLIP encoder, then reorder. If the rescoring encoder == retrieval encoder the
    order is unchanged, so pass a stronger model_key here (e.g. 'max').
    """
    name = "clip_rescore"

    def __init__(self, encoder, image_paths: Sequence[str], caption_texts: Sequence[str]):
        self.enc = encoder
        self.image_paths = list(image_paths)
        self.caption_texts = list(caption_texts)

    def rerank_text_to_image(self, query_text: str, candidate_ids: Sequence[int]) -> List[int]:
        if not candidate_ids:
            return list(candidate_ids)
        q = self.enc.encode_texts([query_text], batch_size=1)            # (1, D)
        imgs = [Image.open(self.image_paths[i]).convert("RGB") for i in candidate_ids]
        cand = self.enc.encode_images(imgs)                              # (C, D)
        sims = (cand @ q[0])
        order = np.argsort(-sims)
        return [candidate_ids[i] for i in order]

    def rerank_image_to_text(self, query_image: Image.Image, candidate_ids: Sequence[int]) -> List[int]:
        if not candidate_ids:
            return list(candidate_ids)
        q = self.enc.encode_images([query_image])                       # (1, D)
        cand = self.enc.encode_texts([self.caption_texts[i] for i in candidate_ids])
        sims = (cand @ q[0])
        order = np.argsort(-sims)
        return [candidate_ids[i] for i in order]


# --------------------------------------------------------------------------- #
class BLIPReranker:
    """
    Cross-encoder reranker using BLIP's image-text-matching (ITM) head
    (`Salesforce/blip-itm-base-coco`). For each (image, text) candidate pair it
    returns P(match); we sort candidates by that probability.
    """
    name = "blip_itm"

    def __init__(self, image_paths: Sequence[str], caption_texts: Sequence[str],
                 model_name: str = "Salesforce/blip-itm-base-coco",
                 device: str | None = None, batch_size: int = 32):
        import torch
        from transformers import BlipForImageTextRetrieval, BlipProcessor
        from utils import suppress_stderr
        self.torch = torch
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        with suppress_stderr():
            self.processor = BlipProcessor.from_pretrained(model_name)
            self.model = BlipForImageTextRetrieval.from_pretrained(model_name).to(self.device).eval()
        self.image_paths = list(image_paths)
        self.caption_texts = list(caption_texts)
        self.bs = batch_size

    def _itm_scores(self, images, texts) -> np.ndarray:
        """P(match) for paired lists images[i] <-> texts[i]."""
        torch = self.torch
        scores = []
        for s in range(0, len(images), self.bs):
            imgs = images[s:s + self.bs]
            txts = texts[s:s + self.bs]
            inp = self.processor(images=imgs, text=txts, return_tensors="pt",
                                 padding=True, truncation=True).to(self.device)
            with torch.no_grad():
                out = self.model(**inp, use_itm_head=True)
                prob = torch.softmax(out.itm_score, dim=1)[:, 1]  # col 1 = "matched"
            scores.append(prob.float().cpu().numpy())
        return np.concatenate(scores, 0)

    def rerank_text_to_image(self, query_text: str, candidate_ids: Sequence[int]) -> List[int]:
        if not candidate_ids:
            return list(candidate_ids)
        imgs = [Image.open(self.image_paths[i]).convert("RGB") for i in candidate_ids]
        scores = self._itm_scores(imgs, [query_text] * len(candidate_ids))
        order = np.argsort(-scores)
        return [candidate_ids[i] for i in order]

    def rerank_image_to_text(self, query_image: Image.Image, candidate_ids: Sequence[int]) -> List[int]:
        if not candidate_ids:
            return list(candidate_ids)
        txts = [self.caption_texts[i] for i in candidate_ids]
        scores = self._itm_scores([query_image.convert("RGB")] * len(candidate_ids), txts)
        order = np.argsort(-scores)
        return [candidate_ids[i] for i in order]


def make_reranker(kind: str, **kwargs):
    return {"identity": IdentityReranker,
            "clip_rescore": CLIPRescoreReranker,
            "blip_itm": BLIPReranker}[kind](**kwargs)


In [ ]:
%%writefile src/pipeline.py
"""
pipeline.py
===========
TwoStageRetriever = bi-encoder retrieval (stage 1) + optional reranking (stage 2).

    stage 1:  encode query -> FAISS top-`depth` candidates           (fast, recall-y)
    stage 2:  reranker re-scores those `depth` candidates            (slow, precise)

This is the standard retrieve-then-rerank pattern used in search and RAG. It is
deliberately model-agnostic: it depends only on the Encoder / VectorIndex /
Reranker interfaces, so you can drop in a bigger CLIP, an ANN index, or a BLIP
cross-encoder without changing this file.
"""
from __future__ import annotations

from dataclasses import dataclass
from typing import List, Optional, Sequence, Tuple

import numpy as np
from PIL import Image

from interfaces import Encoder, Reranker, VectorIndex


@dataclass
class RetrievalResult:
    ids: List[int]
    scores: List[float]
    payloads: List[str]   # image path (t2i) or caption text (i2t)


class TwoStageRetriever:
    def __init__(self,
                 encoder: Encoder,
                 image_index: VectorIndex,
                 text_index: VectorIndex,
                 image_paths: Sequence[str],
                 caption_texts: Sequence[str],
                 reranker: Optional[Reranker] = None):
        self.encoder = encoder
        self.image_index = image_index
        self.text_index = text_index
        self.image_paths = list(image_paths)      # aligned to image gallery ids
        self.caption_texts = list(caption_texts)  # aligned to text gallery ids
        self.reranker = reranker

    # ---------- serving ----------
    def search_text_to_image(self, text: str, k: int = 5,
                             depth: int = 50, rerank: bool = True) -> RetrievalResult:
        q = self.encoder.encode_texts([text], batch_size=1)
        scores, idx = self.image_index.search(q, max(k, depth if rerank else k))
        cand = idx[0].tolist()
        sc = scores[0].tolist()
        if rerank and self.reranker is not None:
            order = self.reranker.rerank_text_to_image(text, cand[:depth])
            cand = order + cand[depth:]
            sc = [float("nan")] * len(order) + sc[depth:]
        cand, sc = cand[:k], sc[:k]
        return RetrievalResult(cand, sc, [self.image_paths[i] for i in cand])

    def search_image_to_text(self, image: Image.Image, k: int = 5,
                             depth: int = 50, rerank: bool = True) -> RetrievalResult:
        q = self.encoder.encode_images([image], batch_size=1)
        scores, idx = self.text_index.search(q, max(k, depth if rerank else k))
        cand = idx[0].tolist()
        sc = scores[0].tolist()
        if rerank and self.reranker is not None:
            order = self.reranker.rerank_image_to_text(image, cand[:depth])
            cand = order + cand[depth:]
            sc = [float("nan")] * len(order) + sc[depth:]
        cand, sc = cand[:k], sc[:k]
        return RetrievalResult(cand, sc, [self.caption_texts[i] for i in cand])


def splice_reranked(full_row: np.ndarray, reordered_head: Sequence[int]) -> np.ndarray:
    """
    Replace the first len(reordered_head) entries of a full ranking with the
    reranked order, keeping the stage-1 tail. Used by the evaluation harness so
    reranked metrics still have a complete ranking for MRR/mAP/nDCG.
    """
    head_set = set(reordered_head)
    tail = [j for j in full_row if j not in head_set]
    return np.array(list(reordered_head) + tail, dtype=full_row.dtype)


In [ ]:
%%writefile src/evaluation.py
"""
evaluation.py
=============
Benchmark harness. Produces, for both retrieval directions:
  * Recall@1/5/10, MRR, mAP, nDCG@10 with 95% bootstrap confidence intervals
  * an optional reranked variant (stage-2) evaluated on a query subsample
  * a paired bootstrap significance test (does reranking actually help?)

It operates on precomputed embeddings so stage-1 is cheap; the (expensive)
reranker is only invoked on the subsample used for the reranked numbers.
"""
from __future__ import annotations

from collections import defaultdict
from typing import Callable, Dict, List, Optional, Sequence

import numpy as np

from indexes import make_index
from metrics import aggregate, per_query_metrics
from pipeline import splice_reranked


def _targets_text_to_image(caption_imgidx: Sequence[int]) -> List[set]:
    return [{caption_imgidx[i]} for i in range(len(caption_imgidx))]


def _targets_image_to_text(caption_imgidx: Sequence[int], n_images: int) -> List[set]:
    m = defaultdict(set)
    for crow, ipos in enumerate(caption_imgidx):
        m[ipos].add(crow)
    return [m[i] for i in range(n_images)]


def _full_ranking(index, query_emb: np.ndarray) -> np.ndarray:
    _, ranked = index.search(query_emb, index.ntotal)
    return ranked


def evaluate_direction(query_emb: np.ndarray,
                       gallery_emb: np.ndarray,
                       target_sets: List[set],
                       ks=(1, 5, 10),
                       with_map=True,
                       index_kind="flat_ip",
                       n_boot=1000,
                       seed=0) -> Dict:
    """Baseline (stage-1 only) evaluation for one direction."""
    index = make_index(index_kind).build(gallery_emb)
    ranked = _full_ranking(index, query_emb)
    pq = per_query_metrics(ranked, target_sets, ks=ks, with_map=with_map)
    return {"aggregate": aggregate(pq, n_boot=n_boot, seed=seed),
            "per_query": pq, "ranked": ranked, "index": index}


def evaluate_reranked(ranked_stage1: np.ndarray,
                      target_sets: List[set],
                      rerank_fn: Callable[[int, List[int]], List[int]],
                      depth: int = 50,
                      subsample: int = 500,
                      ks=(1, 5, 10),
                      with_map=True,
                      n_boot=1000,
                      seed=0) -> Dict:
    """
    Apply a reranker to the top-`depth` of each (subsampled) query's stage-1
    ranking and recompute metrics.

    rerank_fn(query_row_index, candidate_ids) -> reordered candidate_ids
    """
    rng = np.random.default_rng(seed)
    n = len(ranked_stage1)
    sel = np.arange(n) if subsample >= n else rng.choice(n, size=subsample, replace=False)

    reranked_rows, sub_targets = [], []
    for qi in sel:
        row = ranked_stage1[qi]
        head = row[:depth].tolist()
        reordered = rerank_fn(int(qi), head)
        reranked_rows.append(splice_reranked(row, reordered))
        sub_targets.append(target_sets[qi])
    reranked = np.vstack(reranked_rows)

    pq = per_query_metrics(reranked, sub_targets, ks=ks, with_map=with_map)
    return {"aggregate": aggregate(pq, n_boot=n_boot, seed=seed),
            "per_query": pq, "n_queries": len(sel), "selected": sel}


def paired_bootstrap_delta(before: np.ndarray, after: np.ndarray,
                           n_boot=1000, seed=0) -> Dict:
    """
    Paired bootstrap on the SAME queries: is mean(after) - mean(before) > 0?
    Returns the mean delta and a 95% CI; CI excluding 0 => significant.
    """
    before, after = np.asarray(before), np.asarray(after)
    assert before.shape == after.shape
    rng = np.random.default_rng(seed)
    n = len(before)
    idx = rng.integers(0, n, size=(n_boot, n))
    deltas = after[idx].mean(1) - before[idx].mean(1)
    lo, hi = np.percentile(deltas, [2.5, 97.5])
    return {"delta": round(float((after - before).mean()) * 100, 2),
            "lo": round(float(lo) * 100, 2), "hi": round(float(hi) * 100, 2),
            "significant": bool(lo > 0 or hi < 0)}


def run_full_benchmark(img_emb, txt_emb, caption_imgidx,
                       index_kind="flat_ip", ks=(1, 5, 10), seed=0) -> Dict:
    """Both directions, baseline only. Returns a tidy results dict."""
    t2i = evaluate_direction(txt_emb, img_emb,
                             _targets_text_to_image(caption_imgidx),
                             ks=ks, with_map=False, index_kind=index_kind, seed=seed)
    i2t = evaluate_direction(img_emb, txt_emb,
                             _targets_image_to_text(caption_imgidx, img_emb.shape[0]),
                             ks=ks, with_map=True, index_kind=index_kind, seed=seed)
    return {"text_to_image": t2i, "image_to_text": i2t}


In [ ]:
%%writefile src/data.py
"""
data.py
=======
Flickr8k data module: parse captions, keep images that exist, split BY IMAGE
(so no caption leaks across train/test), and flatten the test split into the
parallel arrays the pipeline consumes.
"""
from __future__ import annotations

import os
import random
from dataclasses import dataclass, field
from typing import Dict, List, Tuple


@dataclass
class DataModule:
    data_dir: str
    images_sub: str = "Images"
    captions_file: str = "captions.txt"
    image_to_captions: Dict[str, List[str]] = field(default_factory=dict)
    image_paths: Dict[str, str] = field(default_factory=dict)

    def load(self, max_images: int | None = None) -> "DataModule":
        images_dir = os.path.join(self.data_dir, self.images_sub)
        caps_path = os.path.join(self.data_dir, self.captions_file)
        assert os.path.isdir(images_dir), f"No Images/ at {images_dir}"
        assert os.path.isfile(caps_path), f"No captions at {caps_path}"

        with open(caps_path, encoding="utf-8") as f:
            lines = f.readlines()
        start = 1 if lines and lines[0].lower().startswith("image") else 0
        raw: Dict[str, List[str]] = {}
        for ln in lines[start:]:
            ln = ln.rstrip("\n")
            if not ln.strip():
                continue
            parts = ln.split(",", 1)          # first comma only
            if len(parts) == 2:
                raw.setdefault(parts[0].strip(), []).append(parts[1].strip())

        for img, caps in raw.items():
            p = os.path.join(images_dir, img)
            if os.path.isfile(p):
                self.image_to_captions[img] = caps
                self.image_paths[img] = os.path.abspath(p)

        if max_images:
            keep = sorted(self.image_to_captions)[:max_images]
            self.image_to_captions = {k: self.image_to_captions[k] for k in keep}
            self.image_paths = {k: self.image_paths[k] for k in keep}
        return self

    def stats(self) -> dict:
        lens = [len(c.split()) for caps in self.image_to_captions.values() for c in caps]
        n_img = len(self.image_to_captions)
        s = {"images": n_img, "captions": len(lens),
             "caps_per_image": round(len(lens) / max(n_img, 1), 2),
             "avg_caption_len": round(sum(lens) / max(len(lens), 1), 2)}
        print("[data]", s)
        return s

    def split_by_image(self, n_test: int = 1000, seed: int = 42) -> Tuple[List[str], List[str]]:
        ids = sorted(self.image_to_captions)
        random.Random(seed).shuffle(ids)
        n_test = min(n_test, max(1, len(ids) // 5))
        return sorted(ids[n_test:]), sorted(ids[:n_test])

    def flatten(self, image_ids: List[str]):
        """Return (image_ids, image_path_list, caption_texts, caption_imgidx)."""
        path_list = [self.image_paths[i] for i in image_ids]
        texts, imgidx = [], []
        for pos, img in enumerate(image_ids):
            for cap in self.image_to_captions[img]:
                texts.append(cap)
                imgidx.append(pos)
        return image_ids, path_list, texts, imgidx


In [ ]:
%%writefile src/config.py
"""
config.py
=========
Single typed configuration object, loadable from YAML. One place for every knob,
which keeps runs reproducible and makes the system self-documenting.
"""
from __future__ import annotations

from dataclasses import asdict, dataclass


@dataclass
class RunConfig:
    # data
    data_dir: str = "/kaggle/input/flickr8k"
    n_test: int = 1000
    seed: int = 42
    smoke_test: bool = True
    smoke_n: int = 100
    # model / index / rerank
    model_key: str = "best"            # fast | b16 | best | max
    index_kind: str = "flat_ip"        # flat_ip | ivf_flat | hnsw
    reranker: str = "blip_itm"         # identity | clip_rescore | blip_itm
    rerank_depth: int = 50
    rerank_subsample: int = 500        # queries used for the (expensive) reranked eval
    # batching
    img_batch: int = 64
    txt_batch: int = 256
    # eval
    ks: tuple = (1, 5, 10)
    n_boot: int = 1000
    out_dir: str = "/kaggle/working"

    @classmethod
    def from_yaml(cls, path: str) -> "RunConfig":
        import yaml
        with open(path) as f:
            data = yaml.safe_load(f) or {}
        if "ks" in data:
            data["ks"] = tuple(data["ks"])
        return cls(**data)

    def to_dict(self) -> dict:
        return asdict(self)


In [ ]:
%%writefile src/finetune.py
"""
finetune.py
===========
Parameter-efficient fine-tuning of CLIP for retrieval, using LoRA adapters on the
attention projections of BOTH the vision and text towers (PEFT). Only the adapters
train (<1% of parameters); the base weights stay frozen.

Objective = CLIP's own symmetric contrastive (InfoNCE) loss over a batch of N
distinct (image, caption) pairs: the N x N similarity matrix should be largest on
its diagonal. We sample ONE caption per image per step so a batch never contains
two captions of the same image (which would create false negatives).

Usage (on GPU):
    enc = CLIPEncoder("fast")                      # baseline encoder
    # ... evaluate baseline ...
    enc = train_lora(enc, train_ids, image_paths, image_to_captions, epochs=2)
    # enc now produces LoRA-adapted embeddings; re-embed + re-evaluate.

The contrastive-loss math is unit-tested (numpy reference) in tests/.
"""
from __future__ import annotations

import random
from typing import Dict, List, Sequence

import numpy as np


# --------------------------------------------------------------------------- #
# Loss (torch imported lazily so the module loads without torch present)
# --------------------------------------------------------------------------- #
def clip_contrastive_loss(image_embeds, text_embeds, logit_scale):
    """
    Symmetric InfoNCE. image_embeds/text_embeds are L2-normalized (N, D);
    logit_scale is a scalar tensor. Returns a scalar loss.
    """
    import torch
    import torch.nn.functional as F
    logits = logit_scale * image_embeds @ text_embeds.t()      # (N, N)
    labels = torch.arange(image_embeds.size(0), device=image_embeds.device)
    return 0.5 * (F.cross_entropy(logits, labels) + F.cross_entropy(logits.t(), labels))


# --------------------------------------------------------------------------- #
# Dataset
# --------------------------------------------------------------------------- #
def build_pairs(train_ids: Sequence[str],
                image_paths: Dict[str, str],
                image_to_captions: Dict[str, List[str]]):
    """List of (image_path, [captions]) for the training images."""
    return [(image_paths[i], image_to_captions[i]) for i in train_ids]


def _make_dataset(pairs, processor, seed=0):
    import torch
    from PIL import Image

    class PairDataset(torch.utils.data.Dataset):
        def __init__(self):
            self.pairs = pairs
            self.rng = random.Random(seed)

        def __len__(self):
            return len(self.pairs)

        def __getitem__(self, idx):
            path, caps = self.pairs[idx]
            return Image.open(path).convert("RGB"), self.rng.choice(caps)

    def collate(batch):
        imgs, caps = zip(*batch)
        pix = processor(images=list(imgs), return_tensors="pt")["pixel_values"]
        tok = processor(text=list(caps), return_tensors="pt", padding=True, truncation=True)
        return pix, tok["input_ids"], tok["attention_mask"]

    return PairDataset(), collate


# --------------------------------------------------------------------------- #
# LoRA wiring + training
# --------------------------------------------------------------------------- #
def apply_lora(clip_model, r=8, alpha=16, dropout=0.05,
               targets=("q_proj", "k_proj", "v_proj", "out_proj")):
    """Inject LoRA adapters into both encoders' attention projections."""
    from peft import LoraConfig, get_peft_model
    cfg = LoraConfig(r=r, lora_alpha=alpha, lora_dropout=dropout,
                     target_modules=list(targets), bias="none")
    peft_model = get_peft_model(clip_model, cfg)
    peft_model.print_trainable_parameters()
    return peft_model


def train_lora(encoder,
               train_ids: Sequence[str],
               image_paths: Dict[str, str],
               image_to_captions: Dict[str, List[str]],
               epochs: int = 2,
               lr: float = 1e-4,
               batch_size: int = 128,
               r: int = 8,
               seed: int = 0,
               save_dir: str | None = None):
    """
    Fine-tune `encoder` in place with LoRA. After training, encoder.encode_images/
    encode_texts produce adapted embeddings. Returns the same encoder.
    """
    import torch
    import torch.nn.functional as F
    from tqdm.auto import tqdm

    device = encoder.device
    peft_model = apply_lora(encoder.model, r=r)
    clip = peft_model.base_model.model           # underlying CLIPModel (adapters injected)
    peft_model.to(device).train()

    pairs = build_pairs(train_ids, image_paths, image_to_captions)
    ds, collate = _make_dataset(pairs, encoder.processor, seed=seed)
    loader = torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=True,
                                         collate_fn=collate, num_workers=2, drop_last=True)

    params = [p for p in peft_model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr)

    if len(loader) == 0:
        print("  (training set too small for this batch_size — skipping LoRA training)")
        peft_model.eval()
        encoder.model = clip
        return encoder

    for ep in range(epochs):
        running, n = 0.0, 0
        for pix, ids, mask in tqdm(loader, desc=f"LoRA epoch {ep+1}/{epochs}"):
            pix, ids, mask = pix.to(device), ids.to(device), mask.to(device)
            with torch.autocast(device):
                _vout = clip.vision_model(pixel_values=pix)
                _tout = clip.text_model(input_ids=ids, attention_mask=mask)
                _img = clip.visual_projection(_vout.pooler_output)
                _txt = clip.text_projection(_tout.pooler_output)
                img = F.normalize(_img, dim=-1)
                txt = F.normalize(_txt, dim=-1)
                scale = clip.logit_scale.exp().clamp(max=100.0)
                loss = clip_contrastive_loss(img, txt, scale)
            opt.zero_grad()
            loss.backward()
            opt.step()
            running += float(loss.item())
            n += 1
        print(f"  epoch {ep+1}: mean contrastive loss = {running/max(n,1):.4f}")

    peft_model.eval()
    if save_dir:
        peft_model.save_pretrained(save_dir)
        print(f"  saved LoRA adapter -> {save_dir}")
    # Route the encoder's feature calls through the adapted CLIPModel.
    encoder.model = clip
    return encoder


### Sanity-check the metric logic before any GPU spend

In [ ]:
sys.path.insert(0, "src")
# Drop any stale cached versions so re-running the %%writefile cells above always
# takes effect WITHOUT a kernel restart (Python caches imported modules otherwise).
for _m in ["utils","interfaces","metrics","indexes","encoders","rerankers",
           "pipeline","evaluation","data","config","finetune"]:
    sys.modules.pop(_m, None)
import numpy as np, evaluation as E
img = np.eye(6, 32, dtype="float32"); img/=np.linalg.norm(img,axis=1,keepdims=True)
txt = np.repeat(img,5,0); cidx=[i//5 for i in range(30)]
r = E.run_full_benchmark(img, txt, cidx)
assert r["image_to_text"]["aggregate"]["mAP"]["mean"]==100.0
print("metric self-check PASSED")


## 2 · Config

In [ ]:
from config import RunConfig
cfg = RunConfig(
    data_dir="/kaggle/input/datasets/adityajn105/flickr8k",
    model_key="best",      # fast | b16 | best | max  (biggest lever on score)
    index_kind="flat_ip",
    reranker="blip_itm",   # identity | clip_rescore | blip_itm
    rerank_depth=50, rerank_subsample=500,
    smoke_test=False, smoke_n=100, n_test=1000, out_dir="/kaggle/working",
)
print(cfg)


## 3 · Data (load, split by image, flatten test set)

In [ ]:
from data import DataModule
from PIL import Image
dm = DataModule(cfg.data_dir).load(max_images=cfg.smoke_n if cfg.smoke_test else None)
dm.stats()
n_test = cfg.smoke_n//2 if cfg.smoke_test else cfg.n_test
train_ids, test_ids = dm.split_by_image(n_test=n_test, seed=cfg.seed)
image_ids, image_path_list, caption_texts, caption_imgidx = dm.flatten(test_ids)
print(f"train images={len(train_ids)}  test images={len(image_ids)}  test captions={len(caption_texts)}")


## 4 · Encode (CLIP bi-encoder)

In [ ]:
from encoders import CLIPEncoder
import time
enc = CLIPEncoder(cfg.model_key)
t0=time.time()
img_emb = enc.encode_images([Image.open(p) for p in image_path_list], cfg.img_batch)
txt_emb = enc.encode_texts(caption_texts, cfg.txt_batch)
print(f"img_emb={img_emb.shape}  txt_emb={txt_emb.shape}  in {time.time()-t0:.1f}s")


## 5 · Baseline benchmark (stage-1) with 95% bootstrap CIs

In [ ]:
import pandas as pd
def table(agg_map):
    rows=[]
    for name,agg in agg_map.items():
        row={"setting":name}
        for k,v in agg.items(): row[k]=f"{v['mean']:.2f} [{v['lo']:.1f},{v['hi']:.1f}]"
        rows.append(row)
    return pd.DataFrame(rows).set_index("setting")
base = E.run_full_benchmark(img_emb, txt_emb, caption_imgidx, index_kind=cfg.index_kind, seed=cfg.seed)
base_tbl = table({"Text→Image": base["text_to_image"]["aggregate"],
                  "Image→Text": base["image_to_text"]["aggregate"]})
print(base_tbl.to_string())


## 6 · Index study — exact vs approximate (justify the choice)

In [ ]:
import time as _t
from indexes import make_index
def measure(kind):
    idx = make_index(kind).build(img_emb)
    t0=_t.time(); _,got = idx.search(txt_emb, 10); dt=(_t.time()-t0)*1000
    _,exact = make_index("flat_ip").build(img_emb).search(txt_emb,10)
    rec = np.mean([len(set(got[i])&set(exact[i]))/10 for i in range(len(txt_emb))])
    return {"index":kind, "recall@10_vs_exact":round(float(rec),3), "search_ms":round(dt,1)}
print(pd.DataFrame([measure(k) for k in ["flat_ip","ivf_flat","hnsw"]]).set_index("index").to_string())


## 7 · Stage-2 rerank + reranked benchmark + paired significance test

In [ ]:
from rerankers import make_reranker
from pipeline import TwoStageRetriever
rk = dict(image_paths=image_path_list, caption_texts=caption_texts)
if cfg.reranker=="clip_rescore": rk["encoder"]=CLIPEncoder("max")
reranker = make_reranker(cfg.reranker, **rk) if cfg.reranker!="identity" else None
retriever = TwoStageRetriever(enc, make_index("flat_ip").build(img_emb),
                              make_index("flat_ip").build(txt_emb),
                              image_path_list, caption_texts, reranker)

if reranker is not None:
    t2i_t = E._targets_text_to_image(caption_imgidx)
    rr_t2i = E.evaluate_reranked(base["text_to_image"]["ranked"], t2i_t,
                rerank_fn=lambda qi,c: reranker.rerank_text_to_image(caption_texts[qi], c),
                depth=cfg.rerank_depth, subsample=cfg.rerank_subsample, with_map=False, seed=cfg.seed)
    i2t_t = E._targets_image_to_text(caption_imgidx, img_emb.shape[0])
    rr_i2t = E.evaluate_reranked(base["image_to_text"]["ranked"], i2t_t,
                rerank_fn=lambda qi,c: reranker.rerank_image_to_text(Image.open(image_path_list[qi]), c),
                depth=cfg.rerank_depth, subsample=cfg.rerank_subsample, with_map=True, seed=cfg.seed)
    print(table({"Text→Image (rerank)": rr_t2i["aggregate"],
                 "Image→Text (rerank)": rr_i2t["aggregate"]}).to_string(), "\n")
    for tag,bpq,rr in [("T→I",base["text_to_image"]["per_query"],rr_t2i),
                       ("I→T",base["image_to_text"]["per_query"],rr_i2t)]:
        d=E.paired_bootstrap_delta(bpq["R@1"][rr["selected"]], rr["per_query"]["R@1"], seed=cfg.seed)
        print(f"rerank {tag} R@1 delta {d['delta']:+.2f} [{d['lo']},{d['hi']}]  significant={d['significant']}")
else:
    print("reranker=identity (baseline only)")


## 8 · LoRA fine-tuning — parameter-efficient adaptation
Zero-shot CLIP isn't tuned to Flickr8k. We inject **LoRA adapters** into the
attention projections of both towers (training <1% of parameters) and optimize
CLIP's **contrastive loss** on the *train* split, then re-evaluate the *test* split
and test whether R@1 improved (paired bootstrap).

Run on a compact backbone (`fast` = ViT-B/32) so it trains in minutes on a T4 and
the before/after is on the *same* model. Bump `LORA_BASE` to `b16`/`best` for a
stronger result if you have GPU headroom (lower the batch size).

In [ ]:
from finetune import train_lora
LORA_BASE   = "fast"
LORA_EPOCHS = 3 if not cfg.smoke_test else 2
LORA_BS     = 128 if not cfg.smoke_test else 8

enc_l = CLIPEncoder(LORA_BASE)                       # same-backbone baseline
imgL0 = enc_l.encode_images([Image.open(p) for p in image_path_list], cfg.img_batch)
txtL0 = enc_l.encode_texts(caption_texts, cfg.txt_batch)
base_l = E.run_full_benchmark(imgL0, txtL0, caption_imgidx, seed=cfg.seed)

enc_l = train_lora(enc_l, train_ids, dm.image_paths, dm.image_to_captions,
                   epochs=LORA_EPOCHS, lr=1e-4, batch_size=LORA_BS, seed=cfg.seed,
                   save_dir=f"{cfg.out_dir}/lora_adapter")

imgL1 = enc_l.encode_images([Image.open(p) for p in image_path_list], cfg.img_batch)
txtL1 = enc_l.encode_texts(caption_texts, cfg.txt_batch)
ft_l = E.run_full_benchmark(imgL1, txtL1, caption_imgidx, seed=cfg.seed)


In [ ]:
print(table({f"{LORA_BASE} zero-shot  T→I": base_l["text_to_image"]["aggregate"],
             f"{LORA_BASE} + LoRA      T→I": ft_l["text_to_image"]["aggregate"],
             f"{LORA_BASE} zero-shot  I→T": base_l["image_to_text"]["aggregate"],
             f"{LORA_BASE} + LoRA      I→T": ft_l["image_to_text"]["aggregate"]}).to_string(), "\n")
lora_deltas = {}
for tag,b,f in [("T→I",base_l["text_to_image"],ft_l["text_to_image"]),
                ("I→T",base_l["image_to_text"],ft_l["image_to_text"])]:
    d=E.paired_bootstrap_delta(b["per_query"]["R@1"], f["per_query"]["R@1"], seed=cfg.seed)
    lora_deltas[tag] = d
    print(f"LoRA {tag} R@1 delta {d['delta']:+.2f} [{d['lo']},{d['hi']}]  significant={d['significant']}")

## 9 · Qualitative examples

In [ ]:
import matplotlib.pyplot as plt, random
random.seed(cfg.seed)
queries = random.sample(caption_texts, 3)
fig, axes = plt.subplots(3,3, figsize=(11,11))
for r,qq in enumerate(queries):
    res = retriever.search_text_to_image(qq, k=3, rerank=reranker is not None)
    for c,p in enumerate(res.payloads):
        axes[r][c].imshow(Image.open(p)); axes[r][c].axis("off"); axes[r][c].set_title(f"#{c+1}",fontsize=9)
    axes[r][0].text(-0.04,0.5,f"Q: {qq[:34]}",rotation=90,va="center",ha="right",
                    transform=axes[r][0].transAxes, fontsize=9)
plt.suptitle("Text → Image (two-stage)"); plt.tight_layout()
plt.savefig(f"{cfg.out_dir}/qualitative_examples.png", dpi=130, bbox_inches="tight"); plt.show()


## 10 · Interactive demo (Gradio) — toggle rerank live
`share=True` gives a public link. Set `share=False` if you Save Version (commit).

In [ ]:
import gradio as gr
def _t2i(q, do_rerank, k):
    res = retriever.search_text_to_image(q, k=int(k), rerank=do_rerank)
    return [Image.open(p) for p in res.payloads]
with gr.Blocks() as demo:
    gr.Markdown("### CLIP + FAISS + BLIP — text→image")
    q=gr.Textbox(value="a dog running on the beach", label="query")
    rr=gr.Checkbox(value=reranker is not None, label="stage-2 rerank")
    k=gr.Slider(1,10,value=5,step=1,label="top-K")
    g=gr.Gallery(columns=5, height=240)
    gr.Button("Search").click(_t2i,[q,rr,k],g)
demo.launch(share=True, debug=False)


## 11 · Save artifacts

In [ ]:
import json
base_tbl.to_csv(f"{cfg.out_dir}/metrics_baseline.csv")
np.save(f"{cfg.out_dir}/img_emb.npy", img_emb); np.save(f"{cfg.out_dir}/txt_emb.npy", txt_emb)
summary={"config":cfg.to_dict(),
         "baseline":{"text_to_image":base["text_to_image"]["aggregate"],
                     "image_to_text":base["image_to_text"]["aggregate"]},
         "lora":{"backbone":LORA_BASE,
                 "epochs":LORA_EPOCHS,
                 "zero_shot":{"text_to_image":base_l["text_to_image"]["aggregate"],
                              "image_to_text":base_l["image_to_text"]["aggregate"]},
                 "fine_tuned":{"text_to_image":ft_l["text_to_image"]["aggregate"],
                               "image_to_text":ft_l["image_to_text"]["aggregate"]},
                 "r1_delta":lora_deltas}}
json.dump(summary, open(f"{cfg.out_dir}/summary.json","w"), indent=2, default=str)
print("Saved to /kaggle/working: metrics_baseline.csv, summary.json, "
      "qualitative_examples.png, img_emb.npy, txt_emb.npy, lora_adapter/")
print("Full-dataset run complete. (For a quick sanity pass, set cfg.smoke_test=True.)")